In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [2]:
class LocalShareClient:

    def __init__(self, user="postgres", password="123456", host="localhost", port="5432"):
        db_url = f"postgresql://{user}:{password}@{host}:{port}/findata"
        self.engine = create_engine(db_url)

    def stock_zh_a_hist(self, 
            symbol: str,
            period: str = "daily",
            start_date: str = "20000101", 
            end_date: str = "20251231", 
            adjust: str = ""
        ) -> pd.DataFrame:

        # 1. 股票代码标准化 (000001 -> sz000001)
        if symbol.startswith('6'): full_code = f"sh{symbol}"
        elif symbol.startswith(('0', '3')): full_code = f"sz{symbol}"
        elif symbol.startswith(('8', '9', '4')): full_code = f"bj{symbol}"
        else: full_code = symbol

        # 2. 日期格式化适配 Timestamptz
        start_dt = f"{start_date[:4]}-{start_date[4:6]}-{start_date[6:]} 00:00:00"
        end_dt = f"{end_date[:4]}-{end_date[4:6]}-{end_date[6:]} 23:59:59"

        # 3. 确定时间聚合粒度
        bucket_map = {"daily": None, "weekly": "1 week", "monthly": "1 month"}
        bucket_str = bucket_map.get(period)

        # 4. 根据复权类型选择基础字段 (增加昨收价 pre_close 的映射)
        if adjust == "hfq":
            col_o, col_h, col_l, col_c, col_pc = "open_back_adj", "high_back_adj", "low_back_adj", "close_back_adj", "pre_close_back_adj"
        else:
            col_o, col_h, col_l, col_c, col_pc = "open", "high", "low", "close", "pre_close"

        # 5. SQL 核心逻辑构建
        if adjust == "qfq":
            print("提示：不建议使用前复权数据进行策略研究，暂未实现此功能。")
            return pd.DataFrame()
        else:
            # --- 不复权 (nfq) & 后复权 (hfq) 逻辑 ---
            if period == "daily":
                query_sql = f"""
                SELECT 
                    (ts AT TIME ZONE 'Asia/Shanghai')::date AS "日期",
                    :raw_symbol AS "股票代码",
                    {col_o} AS "开盘",
                    {col_c} AS "收盘",
                    {col_h} AS "最高",
                    {col_l} AS "最低",
                    vol AS "成交量",
                    amount AS "成交额",
                    ROUND((({col_h} - {col_l}) / NULLIF({col_pc}, 0) * 100)::numeric, 2) AS "振幅",
                    ROUND((({col_c} - {col_pc}) / NULLIF({col_pc}, 0) * 100)::numeric, 2) AS "涨跌幅",
                    ROUND(({col_c} - {col_pc})::numeric, 2) AS "涨跌额",
                    turnover AS "换手率"
                FROM stock_daily
                WHERE code = :code AND ts BETWEEN :start_dt AND :end_dt
                ORDER BY ts ASC;
                """
            else:
                # 不复权/后复权的周/月线 
                # 使用 CTE (WITH agg AS) 先聚合成周线，再用 LAG 窗口函数计算衍生指标
                query_sql = f"""
                    WITH agg AS (
                        SELECT 
                            (MAX(ts) AT TIME ZONE 'Asia/Shanghai')::date AS ts,
                            first({col_o}, ts) AS open,
                            max({col_h}) AS high,
                            min({col_l}) AS low,
                            last({col_c}, ts) AS close,
                            sum(vol) AS vol,
                            sum(amount) AS amount,
                            first({col_pc}, ts) AS pre_close, -- 核心：获取该周期第一天的昨收价
                            sum(turnover) AS turnover
                        FROM stock_daily
                        WHERE code = :code AND ts BETWEEN :start_dt AND :end_dt
                        GROUP BY time_bucket('{bucket_str}', ts AT TIME ZONE 'Asia/Shanghai') 
                    )
                    SELECT 
                        ts AS "日期",
                        :raw_symbol AS "股票代码",
                        open AS "开盘",
                        close AS "收盘",
                        high AS "最高",
                        low AS "最低",
                        vol AS "成交量",
                        amount AS "成交额",
                        -- 核心修正：使用 COALESCE(LAG(close), pre_close) 作为参照基准价
                        ROUND(((high - low) / NULLIF(COALESCE(LAG(close) OVER (ORDER BY ts ASC), pre_close), 0) * 100)::numeric, 2) AS "振幅",
                        ROUND(((close - COALESCE(LAG(close) OVER (ORDER BY ts ASC), pre_close)) / NULLIF(COALESCE(LAG(close) OVER (ORDER BY ts ASC), pre_close), 0) * 100)::numeric, 2) AS "涨跌幅",
                        ROUND((close - COALESCE(LAG(close) OVER (ORDER BY ts ASC), pre_close))::numeric, 2) AS "涨跌额",
                        turnover AS "换手率"
                    FROM agg
                    ORDER BY ts ASC;
                    """

        # 6. 统一执行查询
        with self.engine.connect() as conn:
            df = pd.read_sql(text(query_sql), conn, params={
                "code": full_code,
                "raw_symbol": symbol, # akshare 输出的是无后缀代码
                "start_dt": start_dt,
                "end_dt": end_dt
            })
        
        # 7. 强制对齐 akshare 的数据类型要求
        if not df.empty:
            # 日期和股票代码转为 object (字符串)
            df['日期'] = df['日期'].astype(str)
            df['股票代码'] = df['股票代码'].astype(str)
            
            # 价格及比率等转为 float64
            float_cols = ['开盘', '收盘', '最高', '最低', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']
            for col in float_cols:
                if col in df.columns:
                    df[col] = df[col].astype('float64')
            
            # 成交量转为 int64
            if '成交量' in df.columns:
                df['成交量'] = df['成交量'].astype('int64')

        return df

In [3]:
client = LocalShareClient()
df = client.stock_zh_a_hist(symbol="600015", period="daily", start_date="20050101", end_date="20251231", adjust="nfq")

In [4]:
df

,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
0,2005-01-04,600015,4.14,4.10,4.16,4.07,51998,21361016.0,2.17,-1.20,-0.05,0.43
1,2005-01-05,600015,4.08,4.11,4.17,4.08,35810,14723897.0,2.20,0.24,0.01,0.30
2,2005-01-06,600015,4.12,4.06,4.15,4.04,51993,21167513.0,2.68,-1.22,-0.05,0.43
3,2005-01-07,600015,4.06,4.08,4.13,4.04,33222,13537799.0,2.22,0.49,0.02,0.28
4,2005-01-10,600015,4.08,4.17,4.20,4.07,43390,17981910.0,3.19,2.21,0.09,0.36
...,...,...,...,...,...,...,...,...,...,...,...,...
5004,2025-12-25,600015,6.84,6.85,6.88,6.83,312567,214272656.0,0.73,0.15,0.01,0.20
5005,2025-12-26,600015,6.84,6.82,6.86,6.82,313065,213950816.0,0.58,-0.44,-0.03,0.20
5006,2025-12-29,600015,6.83,6.89,6.89,6.81,538035,369027456.0,1.17,1.03,0.07,0.35
5007,2025-12-30,600015,6.88,6.85,6.89,6.83,363603,249237952.0,0.87,-0.58,-0.04,0.24
